# Run E: mass density ratio for best model vs last model

This notebook produces **two PNG files**. Each PNG has a `(2, 1)` vertical
column of two mass-density-ratio panels:

- **top panel:** J/psi mass density ratio
- **bottom panel:** Z mass density ratio

One PNG is made for `best_model.pt` and one for `last_model.pt`.

The mass panels are the paper-style x-space mass density ratio + residual
panels produced by `scripts_joint/plot_joint_paperstyle.py`.


In [ ]:
# --- setup ---------------------------------------------------------------
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import subprocess
import sys
import shutil

%matplotlib inline

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "scaffold").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scaffold"
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scaffold" else NOTEBOOK_DIR

RUN_DIR = REPO_ROOT / "outputs" / "cms_Joint" / "Run_E"
CONFIG_PATH = REPO_ROOT / "configs_joint" / "cms_Joint_runE.yaml"
PAPERSTYLE_SCRIPT = REPO_ROOT / "scripts_joint" / "plot_joint_paperstyle.py"
FIG_DIR = NOTEBOOK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "best": RUN_DIR / "best_model.pt",
    "last": RUN_DIR / "last_model.pt",
}

# Set this to "zspace" if you want the latent-space mass ratio instead.
MASS_SPACE = "xspace"

print("Repo root:", REPO_ROOT.resolve())
print("Run dir:", RUN_DIR.resolve())
print("Mass space:", MASS_SPACE)


In [ ]:
# --- run paper-style plot generation for each checkpoint ------------------

def run_paperstyle(label, checkpoint_path, output_dir):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(PAPERSTYLE_SCRIPT),
        "--config", str(CONFIG_PATH),
        "--checkpoint", str(checkpoint_path),
        "--output-dir", str(output_dir),
        "--device", "cpu",
        "--split", "test",
        "--max-events", "15000",
        "--batch-size", "4096",
        "--threads", "4",
        "--interop-threads", "4",
    ]
    print("Running:", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

TMP_ROOT = FIG_DIR / ".paperstyle_tmp"

for label, checkpoint_path in CHECKPOINTS.items():
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
    run_paperstyle(label, checkpoint_path, TMP_ROOT / label)

print("Paper-style plots generated.")


In [ ]:
# --- compose the two requested (2,1) PNGs ---------------------------------

def compose_two_regions(label, output_path):
    jpsi_img = TMP_ROOT / label / "jpsi" / f"paperstyle_{MASS_SPACE}_mass_density_ratio.png"
    z_img = TMP_ROOT / label / "z" / f"paperstyle_{MASS_SPACE}_mass_density_ratio.png"

    for path in (jpsi_img, z_img):
        if not path.exists():
            raise FileNotFoundError(f"Missing mass ratio image: {path}")

    fig, axes = plt.subplots(2, 1, figsize=(8.4, 15.2), constrained_layout=True)

    for ax, path, title in (
        (axes[0], jpsi_img, r"$J/\psi$ mass density ratio"),
        (axes[1], z_img, r"$Z$ mass density ratio"),
    ):
        image = mpimg.imread(path)
        ax.imshow(image)
        ax.set_title(title, fontsize=13)
        ax.axis("off")

    fig.suptitle(f"Run E {label} model — mass density ratio", fontsize=16)
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", output_path)

BEST_OUT = FIG_DIR / "Run_E_best_model_mass_density_ratio.png"
LAST_OUT = FIG_DIR / "Run_E_last_model_mass_density_ratio.png"

compose_two_regions("best", BEST_OUT)
compose_two_regions("last", LAST_OUT)

shutil.rmtree(TMP_ROOT, ignore_errors=True)
print("Done.")
